# Test 07: Edge Cases

Tests for idempotency, force-reinstall, error handling, and other
non-happy-path scenarios.

**What this tests:**
1. Idempotency -- re-running install on an existing environment
2. Force reinstall (`force=True`)
3. Dry run mode
4. Install with no EAI configured (`apply_eai=False`)
5. Config from YAML file
6. Config from preset (r_only.yaml)
7. CLI invocation
8. Error handling -- install with no languages enabled

**Prerequisites:**
- Run Test 01 (R-only) first so we have an existing environment to test against

## 1. Install the Toolkit

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

## 2. Idempotency Test

Re-running install on an already-configured environment should:
- Detect existing micromamba (skip download)
- Detect existing packages (skip conda install or only install missing)
- Complete significantly faster than first run

In [ ]:
import time
from sfnb_multilang import install

start = time.time()
report = install(languages=["r"], apply_eai=False)
elapsed = time.time() - start

print(f"Re-install completed in {elapsed:.1f}s")
print(f"Success: {report.success}")

if elapsed < 120:
    print(f"\nPASS: Re-install was fast ({elapsed:.0f}s < 120s) -- caching works")
else:
    print(f"\nWARN: Re-install took {elapsed:.0f}s -- may not be detecting existing packages")

## 3. Force Reinstall

With `force=True`, the installer should re-download and re-install
everything, ignoring cached state.

In [ ]:
report_force = install(languages=["r"], force=True, apply_eai=False)

print(f"Force reinstall success: {report_force.success}")
print(f"Duration: {report_force.elapsed_seconds:.1f}s")

assert report_force.success, "FAIL: Force reinstall failed"
print("\nPASS: Force reinstall completed")

## 4. Dry Run Mode

Dry run should show what would be installed without actually doing it.

In [ ]:
!python -m sfnb_multilang install --r --scala --dry-run

In [ ]:
# Dry run for all languages
!python -m sfnb_multilang install --all --dry-run

## 5. Install with No EAI

With `apply_eai=False`, Phase 0 should be skipped entirely.

In [ ]:
report_no_eai = install(languages=["r"], apply_eai=False)

print(f"Success: {report_no_eai.success}")
print(f"EAI applied: {report_no_eai.eai_applied}")

assert report_no_eai.eai_applied is False, "FAIL: EAI should not have been applied"
print("\nPASS: EAI skipped when apply_eai=False")

## 6. Install from YAML Config File

Test using a preset config instead of keyword arguments.

In [ ]:
# First, check what configs are available
import os
import importlib.resources

# The configs ship with the package
# In Workspace, you'd upload the config or use an inline dict

# Test with programmatic config loading
from sfnb_multilang.config import load_config, ToolkitConfig

# Write a minimal test config
import yaml
test_config = {
    "languages": {
        "r": {"enabled": True, "conda_packages": ["r-tidyverse"]},
        "scala": {"enabled": False},
        "julia": {"enabled": False},
    },
    "network_rule": {"apply_in_installer": False},
}

with open("test_config.yaml", "w") as f:
    yaml.dump(test_config, f)

cfg = load_config("test_config.yaml")
print(f"Loaded config: R={cfg.r.enabled}, Scala={cfg.scala.enabled}, Julia={cfg.julia.enabled}")
print(f"R packages: {cfg.r.conda_packages}")
assert cfg.r.enabled is True
assert cfg.scala.enabled is False
print("\nPASS: YAML config loading works")

In [ ]:
report_yaml = install(config="test_config.yaml")

print(f"Success: {report_yaml.success}")
print(f"Plugins installed: {list(report_yaml.plugin_results.keys())}")

assert "r" in report_yaml.plugin_results, "FAIL: R should be installed"
assert "scala" not in report_yaml.plugin_results, "FAIL: Scala should NOT be installed"
print("\nPASS: YAML config install works")

## 7. CLI Invocation

Test the command-line interface.

In [ ]:
# Help text
!python -m sfnb_multilang --help

In [ ]:
# Install help
!python -m sfnb_multilang install --help

In [ ]:
# Validate existing installation
!python -m sfnb_multilang validate --config test_config.yaml

## 8. Error Handling -- No Languages Enabled

The installer should raise a clear error if no languages are enabled.

In [ ]:
from sfnb_multilang.exceptions import ToolkitError

try:
    install(languages=[])  # No languages
    print("FAIL: Should have raised ToolkitError")
except ToolkitError as e:
    print(f"Got expected error: {e}")
    print("\nPASS: Clear error when no languages enabled")
except Exception as e:
    print(f"Got unexpected error type: {type(e).__name__}: {e}")

## 9. Verbose Logging

Test that `verbose=True` produces DEBUG-level output.

In [ ]:
report_verbose = install(languages=["r"], verbose=True, apply_eai=False)

print(f"\nSuccess: {report_verbose.success}")
print("Check output above for DEBUG-level messages")

## 10. Cleanup

In [ ]:
import os
for f in ["test_config.yaml", "eai_setup.sql"]:
    if os.path.isfile(f):
        os.remove(f)
        print(f"  Removed {f}")
print("Cleanup done")

## Test Summary

| Test | Expected |
|---|---|
| Idempotency | Re-install fast, all packages detected |
| Force reinstall | Succeeds, takes longer than cached |
| Dry run | Prints plan without installing |
| No EAI | Phase 0 skipped, eai_applied=False |
| YAML config | Loads correctly, installs only enabled languages |
| CLI | Help text, install, validate work |
| No languages | ToolkitError raised |
| Verbose | DEBUG messages in output |